# RAG Hallucination & Grounding Risk Scoring

**Abstract.** Retrieval-augmented generation (RAG) reduces hallucination by grounding answers in retrieved documents, but not all RAG outputs are equally trustworthy. This notebook implements a **multi-signal grounding score** that combines: (1) retrieval relevance (how well the top-k documents match the question), (2) citation coverage (whether the answer cites the retrieved sources), and (3) LLM-as-judge groundedness (whether claims are supported by the context). We map the combined signal to a **three-tier routing** (RELIABLE / UNCERTAIN / UNRELIABLE) suitable for production pipelines where low-confidence answers should be escalated or rejected.

**Prerequisites:** Run `foundations/hugging_face_chromadb_demo.ipynb` and `foundations/gemini_rag_pipeline_demo.ipynb` first so that `../chroma` (ostep collection) and the judge pattern are available.

---

**Connection to hallucination metrics.** In our [Hallucinations](https://github.com/A-Kuo/Hallucinations) repo we detect unreliable outputs via attention entropy and cross-layer KL. Here we address the same goal in the RAG setting: *grounding* is the RAG-specific form of "not hallucinating." Combining retrieval, citation, and judge signals gives a calibrated estimate of whether the model stayed within the sources.

In [ ]:
import os
import re
import unicodedata
import json
from pathlib import Path
from dataclasses import dataclass
import chromadb
from google import genai
from dotenv import load_dotenv

load_dotenv()

DATA_DIR = Path('../data')
TXT_DIR = DATA_DIR / 'txt'
MODEL = 'gemini-2.5-flash'
TEMPERATURE = 0
TOP_K = 5

client = genai.Client(
    vertexai=True,
    project=os.getenv('GCP_PROJECT'),
    location=os.getenv('GCP_LOCATION')
)
chroma_client = chromadb.PersistentClient(path='../chroma')
collection = chroma_client.get_collection(name='ostep')

## 1. Retrieval relevance score

We query ChromaDB for top-k documents and use the **inverse of the mean distance** (or similarity) as a proxy for "how well the corpus supports this question." Low relevance → higher hallucination risk.

In [ ]:
def get_retrieval_score(question: str, k: int = TOP_K) -> tuple[float, list]:
    """Return (relevance_score in [0,1], list of doc ids). Chroma returns distances (lower = better)."""
    result = collection.query(query_texts=[question], n_results=k, include=['distances'])
    distances = result['distances'][0]
    ids = result['ids'][0]
    # Convert distance to similarity: 1 / (1 + d) so 0 distance -> 1, large d -> 0
    similarities = [1.0 / (1.0 + d) for d in distances]
    score = sum(similarities) / len(similarities) if similarities else 0.0
    return score, ids

## 2. Citation coverage

Check whether the model's answer references the retrieved source identifiers (e.g. `chapter_id` or doc ids). We parse `<SOURCE ...>` or explicit chapter names from the structured output when available; otherwise we use a simple heuristic: presence of retrieved chapter substrings in the answer.

In [ ]:
def get_citation_coverage(answer: str, doc_ids: list[str]) -> float:
    """Fraction of retrieved doc_ids that appear in the answer (by chapter_id)."""
    chapter_ids = set()
    for doc_id in doc_ids:
        if '_' in doc_id:
            chapter_ids.add(doc_id.rsplit('_', 1)[0])
        else:
            chapter_ids.add(doc_id)
    if not chapter_ids:
        return 1.0
    answer_lower = answer.lower()
    cited = sum(1 for c in chapter_ids if c.replace('-', ' ') in answer_lower or c in answer)
    return cited / len(chapter_ids)

## 3. LLM-as-judge groundedness

Reuse the judge pattern from the foundations: one Gemini call that returns `{ "is_grounded": bool, "unsupported_claims": [...] }`. We map `is_grounded` to 1.0 and optionally penalize by number of unsupported claims.

In [ ]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize('NFKD', text)
    text = re.sub(r'-\s*\n\s*', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def build_context(doc_ids: list, documents: list) -> str:
    parts = []
    for doc_id, doc_text in zip(doc_ids, documents):
        ch, page = doc_id.rsplit('_', 1) if '_' in doc_id else (doc_id, '0')
        parts.append(f"<SOURCE chapter_id=\"{ch}\" page_number=\"{int(page)+1}\">\n{normalize_text(doc_text)}")
    return "\n\n".join(parts)

def judge_groundedness(question: str, answer: str, context: str) -> tuple[float, list]:
    """Return (grounded_score in [0,1], unsupported_claims)."""
    prompt = f"""You are a strict fact-checker. Given the SOURCES below and the USER question and MODEL answer, decide if every factual claim in the answer is supported by the sources.

SOURCES:
{context}

USER question: {question}
MODEL answer: {answer}

Respond with JSON only: {{ "is_grounded": true/false, "unsupported_claims": ["claim1", ...] }}"""
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=genai.GenerateContentConfig(temperature=TEMPERATURE, response_mime_type='application/json')
    )
    try:
        data = json.loads(response.text)
    except Exception:
        return 0.0, ["Parse error"]
    is_grounded = data.get('is_grounded', False)
    claims = data.get('unsupported_claims', [])
    penalty = min(1.0, len(claims) * 0.2)
    score = 1.0 if is_grounded else 0.0
    score = max(0.0, score - penalty)
    return score, claims

## 4. Combined grounding score and three-tier routing

We combine the three signals with configurable weights (default: retrieval 0.25, citation 0.25, judge 0.50) and map to the same **RELIABLE / UNCERTAIN / UNRELIABLE** tiers used in our attention-based hallucination detector (Hallucinations v1):

- **confidence > 0.75** → RELIABLE (route to user)
- **0.50 ≤ confidence ≤ 0.75** → UNCERTAIN (escalate)
- **confidence < 0.50** → UNRELIABLE (reject)

In [ ]:
@dataclass
class GroundingResult:
    retrieval_score: float
    citation_score: float
    judge_score: float
    combined_score: float
    tier: str  # RELIABLE | UNCERTAIN | UNRELIABLE
    answer: str
    doc_ids: list
    unsupported_claims: list

def compute_grounding_score(
    question: str,
    answer: str,
    doc_ids: list,
    documents: list,
    w_retrieval: float = 0.25,
    w_citation: float = 0.25,
    w_judge: float = 0.50,
) -> GroundingResult:
    retrieval_s, _ = get_retrieval_score(question)
    citation_s = get_citation_coverage(answer, doc_ids)
    context = build_context(doc_ids, documents)
    judge_s, claims = judge_groundedness(question, answer, context)
    combined = w_retrieval * retrieval_s + w_citation * citation_s + w_judge * judge_s
    if combined > 0.75:
        tier = 'RELIABLE'
    elif combined >= 0.50:
        tier = 'UNCERTAIN'
    else:
        tier = 'UNRELIABLE'
    return GroundingResult(
        retrieval_score=retrieval_s,
        citation_score=citation_s,
        judge_score=judge_s,
        combined_score=combined,
        tier=tier,
        answer=answer,
        doc_ids=doc_ids,
        unsupported_claims=claims,
    )

## 5. End-to-end: RAG + grounding score

Run retrieval, generate answer with Gemini, then score and route.

In [ ]:
def rag_with_grounding_score(question: str) -> GroundingResult:
    result = collection.query(query_texts=[question], n_results=TOP_K, include=['documents', 'distances'])
    doc_ids = result['ids'][0]
    documents = result['documents'][0]
    context = build_context(doc_ids, documents)
    prompt = f"""Answer the question using ONLY the following sources. Cite chapter names where relevant.

{context}

Question: {question}"""
    response = client.models.generate_content(model=MODEL, contents=prompt, config=genai.GenerateContentConfig(temperature=TEMPERATURE))
    answer = response.text.strip()
    return compute_grounding_score(question, answer, doc_ids, documents)

# Demo
demo_question = "What is a semaphore and how is it used?"
result = rag_with_grounding_score(demo_question)
print(f"Combined score: {result.combined_score:.3f}")
print(f"Tier: {result.tier}")
print(f"Retrieval / Citation / Judge: {result.retrieval_score:.3f} / {result.citation_score:.3f} / {result.judge_score:.3f}")
if result.unsupported_claims:
    print("Unsupported claims:", result.unsupported_claims)